# F5-TTS VOXE Fine-tuning Guide

This notebook provides a complete workflow for fine-tuning F5-TTS on the VOXE dataset.

## Table of Contents
1. [Environment Setup](#setup)
2. [Dataset Verification](#verify)
3. [Training Configuration](#training)
4. [Fine-tuning](#finetune)
5. [Model Testing](#testing)
6. [Inference Examples](#inference)


## 1. Environment Setup {#setup}


In [ ]:
# Verify GPU availability
import torch
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


In [ ]:
# Verify F5-TTS installation
try:
    from f5_tts.api import F5TTS
    from f5_tts.model.dataset import load_dataset
    from f5_tts.model.utils import get_tokenizer
    print("✓ F5-TTS installed correctly")
except ImportError as e:
    print(f"✗ F5-TTS import failed: {e}")
    print("Run: pip install -e .")


## 2. Dataset Verification {#verify}


In [ ]:
# Check VOXE dataset structure
from pathlib import Path
import json

def check_dataset():
    checks = {
        "Manifests": "data/voxe/manifests",
        "Train manifest": "data/voxe/manifests/train.jsonl",
        "Val manifest": "data/voxe/manifests/val.jsonl",
        "WAV files": "data/voxe/wav24k",
        "Vocab file": "data/voxe_char/vocab.txt",
        "Arrow file": "data/voxe_char/raw.arrow",
        "Duration file": "data/voxe_char/duration.json"
    }
    
    all_ok = True
    for name, path in checks.items():
        p = Path(path)
        if p.exists():
            if p.is_file() and path.endswith('.jsonl'):
                count = sum(1 for _ in open(p))
                print(f"✓ {name}: {count} samples")
            elif p.is_file() and path.endswith('.json'):
                with open(p) as f:
                    data = json.load(f)
                    print(f"✓ {name}: {len(data.get('duration', []))} durations")
            elif p.is_file() and path.endswith('.txt'):
                count = sum(1 for _ in open(p))
                print(f"✓ {name}: {count} tokens")
            else:
                print(f"✓ {name}: Found")
        else:
            print(f"✗ {name}: Not found at {path}")
            all_ok = False
    
    return all_ok

dataset_ok = check_dataset()


In [ ]:
# Training configuration
training_config = {
    "exp_name": "F5TTS_v1_Base",  # Model architecture
    "dataset_name": "voxe",  # Dataset name (determines output directory)
    "learning_rate": 1e-5,
    "batch_size_per_gpu": 800,  # Audio frames per GPU
    "batch_size_type": "frame",  # 'frame' or 'sample'
    "max_samples": 8,  # Max sequences per batch
    "epochs": 2,
    "num_warmup_updates": 100,
    "save_per_updates": 50,
    "keep_last_n_checkpoints": 3,
    "finetune": True,
    "tokenizer": "char"
}

print("Training Configuration:")
for key, value in training_config.items():
    print(f"  {key}: {value}")


## 4. Fine-tuning {#finetune}

**Note:** For actual training, use the CLI command below or uncomment the training code in the next cell.


In [ ]:
# Run training using CLI (recommended)
# This executes the same command as run_training_docker.ps1

import subprocess
import sys

cmd = [
    sys.executable,
    "src/f5_tts/train/finetune_cli.py",
    "--exp_name", training_config["exp_name"],
    "--dataset_name", training_config["dataset_name"],
    "--learning_rate", str(training_config["learning_rate"]),
    "--batch_size_per_gpu", str(training_config["batch_size_per_gpu"]),
    "--batch_size_type", training_config["batch_size_type"],
    "--max_samples", str(training_config["max_samples"]),
    "--epochs", str(training_config["epochs"]),
    "--num_warmup_updates", str(training_config["num_warmup_updates"]),
    "--save_per_updates", str(training_config["save_per_updates"]),
    "--keep_last_n_checkpoints", str(training_config["keep_last_n_checkpoints"]),
    "--finetune",
    "--tokenizer", training_config["tokenizer"]
]

print("Training command:")
print(" ".join(cmd))
print("\nTo run training, uncomment the line below:")
# subprocess.run(cmd)


## 5. Model Testing {#testing}


In [ ]:
# Check for trained checkpoints
checkpoint_dir = Path(f"ckpts/{training_config['dataset_name']}")
if checkpoint_dir.exists():
    checkpoints = list(checkpoint_dir.glob("*.pt")) + list(checkpoint_dir.glob("*.safetensors"))
    if checkpoints:
        print("Available checkpoints:")
        for ckpt in sorted(checkpoints):
            size_mb = ckpt.stat().st_size / (1024 * 1024)
            print(f"  - {ckpt.name} ({size_mb:.1f} MB)")
    else:
        print("No checkpoints found. Train the model first.")
else:
    print("Checkpoint directory not found.")


In [ ]:
# Test inference configuration
inference_config = {
    "model": "F5TTS_v1_Base",
    "ckpt_file": "ckpts/voxe/model_last.pt",
    "vocab_file": "data/voxe_char/vocab.txt",
    "ref_audio": "data/voxe/wav24k/esd/0011/Angry/0011_000351.wav",
    "ref_text": "the nine the eggs, i keep.",
    "gen_text": "This is a test of the fine-tuned F5-TTS model on the VOXE dataset.",
    "output_dir": "tests",
    "output_file": "voxe_test.wav"
}

print("Inference Configuration:")
for key, value in inference_config.items():
    print(f"  {key}: {value}")


## 6. Inference Examples {#inference}

Run inference using the CLI command below:


In [ ]:
# Run inference using CLI
import subprocess

inference_cmd = [
    sys.executable,
    "src/f5_tts/infer/infer_cli.py",
    "--model", inference_config["model"],
    "--ckpt_file", inference_config["ckpt_file"],
    "--vocab_file", inference_config["vocab_file"],
    "--ref_audio", inference_config["ref_audio"],
    "--ref_text", inference_config["ref_text"],
    "--gen_text", inference_config["gen_text"],
    "--output_dir", inference_config["output_dir"],
    "--output_file", inference_config["output_file"]
]

print("Inference command:")
print(" ".join(inference_cmd))
print("\nTo run inference, uncomment the line below:")
# subprocess.run(inference_cmd)
